# Phase 5 - Class Imbalance Handling

This notebook prepares and validates the four training-only imbalance scenarios. S3 balances classes at the majority count, while S4 balances classes at the minority count.

In [ ]:
# Cell 1 - Locate the project root and load the libraries used by this notebook.
from pathlib import Path
import hashlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

In [ ]:
# Cell 2 - Load Phase 4 artifacts and record test-file hashes before any scenario runs.
from src.data_loading import load_config, resolve_project_path

config_path = project_root / 'configs' / 'config.yaml'
config = load_config(config_path)
processed_dir = resolve_project_path(config['paths']['processed_dir'], project_root)
metrics_dir = resolve_project_path(config['paths']['metrics_dir'], project_root)
figures_dir = resolve_project_path(config['paths']['figures_dir'], project_root)
figures_dir.mkdir(parents=True, exist_ok=True)

z_train_path = processed_dir / 'Z_train.npy'
y_train_path = processed_dir / 'y_train.npy'
z_test_path = processed_dir / 'Z_test.npy'
y_test_path = processed_dir / 'y_test.npy'
required_paths = [z_train_path, y_train_path, z_test_path, y_test_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing Phase 4 artifacts: {missing_paths}')

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        while block := file.read(block_size):
            digest.update(block)
    return digest.hexdigest()

test_hashes_before = {
    'Z_test': sha256_file(z_test_path),
    'y_test': sha256_file(y_test_path),
}
z_train = np.load(z_train_path, mmap_mode='r')
y_train = np.load(y_train_path, mmap_mode='r')
print(f'Z_train shape: {z_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Test hashes recorded: {test_hashes_before}')

In [ ]:
# Cell 3 - Inspect the original training distribution before imbalance handling.
class_mapping = config['data']['class_mapping']
index_to_class = {int(index): name for name, index in class_mapping.items()}
labels, counts = np.unique(y_train, return_counts=True)
original_distribution = pd.DataFrame({
    'class_index': labels.astype(int),
    'class_name': [index_to_class[int(label)] for label in labels],
    'count': counts.astype(int),
    'percentage': 100.0 * counts / counts.sum(),
})
display(original_distribution)

In [ ]:
# Cell 4 - Prepare all four scenarios, reusing valid disk-backed artifacts when available.
from src.imbalance import prepare_all_imbalance_scenarios

scenario_reports = prepare_all_imbalance_scenarios(config_path, force=False)
scenario_summary = pd.DataFrame([
    {
        'scenario': report['scenario'],
        'input_rows': report['input_rows'],
        'output_rows': report['output_rows'],
        'sampling_target': report['sampling_target'],
        'uses_sample_weight': report['sample_weight_path'] is not None,
    }
    for report in scenario_reports.values()
])
display(scenario_summary)

In [ ]:
# Cell 5 - Load the canonical before/after distribution logs for comparison.
distribution_frames = []
for report in scenario_reports.values():
    frame = pd.read_csv(report['distribution_path'])
    frame.insert(0, 'scenario', report['scenario'])
    distribution_frames.append(frame)

distribution_comparison = pd.concat(distribution_frames, ignore_index=True)
display(distribution_comparison)

In [ ]:
# Cell 6 - Validate scenario behavior and prove that test artifacts remain byte-identical.
num_classes = int(config['data']['expected_num_classes'])
original_counts = np.bincount(y_train, minlength=num_classes)
s2_weights = np.load(
    scenario_reports['s2_class_weight']['sample_weight_path'],
    mmap_mode='r',
)
s3_labels = np.load(scenario_reports['s3_upsampling']['y_train_path'], mmap_mode='r')
s4_labels = np.load(scenario_reports['s4_downsampling']['y_train_path'], mmap_mode='r')
s3_counts = np.bincount(s3_labels, minlength=num_classes)
s4_counts = np.bincount(s4_labels, minlength=num_classes)

assert scenario_reports['s1_none']['z_train_path'] == str(z_train_path)
assert scenario_reports['s1_none']['y_train_path'] == str(y_train_path)
assert s2_weights.shape == y_train.shape
assert np.isfinite(s2_weights).all() and float(s2_weights.min()) > 0.0
assert np.all(s3_counts == original_counts.max())
assert np.all(s4_counts == original_counts.min())
assert all(not report['test_data_modified'] for report in scenario_reports.values())

test_hashes_after = {
    'Z_test': sha256_file(z_test_path),
    'y_test': sha256_file(y_test_path),
}
assert test_hashes_after == test_hashes_before
print('All Phase 5 validations passed.')
print(f'Test hashes unchanged: {test_hashes_after}')

In [ ]:
# Cell 7 - Visualize post-scenario class counts on a logarithmic scale.
scenario_order = config['imbalance']['scenarios']
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
for axis, scenario in zip(axes.flat, scenario_order):
    frame = distribution_comparison[distribution_comparison['scenario'] == scenario]
    axis.bar(frame['class_name'], frame['after_count'])
    axis.set_title(scenario)
    axis.set_yscale('log')
    axis.set_ylabel('Training samples (log scale)')
    axis.tick_params(axis='x', rotation=45)
    axis.grid(axis='y', alpha=0.25)

fig.suptitle('Class Distribution After Each Imbalance Scenario')
fig.tight_layout()
figure_path = figures_dir / 'imbalance_distribution_comparison.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved figure: {figure_path}')

In [ ]:
# Cell 8 - Summarize the Phase 5 artifacts that will be consumed by LightGBM.
artifact_summary = pd.DataFrame([
    {
        'scenario': report['scenario'],
        'training_features': report['z_train_path'],
        'training_labels': report['y_train_path'],
        'sample_weights': report['sample_weight_path'],
    }
    for report in scenario_reports.values()
])
display(artifact_summary)
print('Phase 5 is ready for Phase 6 LightGBM training.')